# Pazy wing continuous gust frequency response

Obtain the frequency response of the straight Pazy wing to a continuous sinusoidal vertical gust. This gives the wingtip response amplitude per unit gust upwash amplitude at each frequency.

Imports

In [ ]:
from jax import numpy as jnp
from matplotlib import pyplot as plt

from flapjax.aero.flowfields import ConstantFlowField
from flapjax.models.pazy.straight.pazy_wing import generate_pazy_wing
from flapjax.structure.utils import get_solve_dofs

Build the wing at a reference flight condition and obtain the static aeroelastic equilibrium. The freestream velocity is well below the flutter onset for this wing (c.f. the flutter analysis tutorial).

In [ ]:
M = 8  # wing chordwise discretisation
M_STAR = 40  # number of wake panels per strip
U_INF = 18.3  # freestream velocity, m/s
RHO = 1.225  # freestream density, kg/m^3
AOA = jnp.deg2rad(5.0)  # angle of attack

wing = generate_pazy_wing(
    flowfield=ConstantFlowField(
        u_inf=jnp.array((U_INF, 0.0, 0.0)),
        rho=RHO,
        relative_motion=True,
    ),
    aoa=AOA,
    m=M,
    m_star=M_STAR,
    node_multiplier=1,
    skin=True,
)

static_sol = wing.static_solve(prescribed_dofs=tuple(range(6)), horseshoe=False)

Linearise the coupled system about the static equilibrium. The resulting `LinearCoupled` object exposes `frf`, which computes the raw gust-to-structural-response transfer function.

In [ ]:
linear = wing.linearise(
    reference=static_sol, batch_size=4, n_struct_modes=10
)

## Frequency sweep

Sweep the gust frequency and evaluate the raw FRF (`flowfield=None`), i.e. the response per unit (1 m/s) gust velocity amplitude at each frequency.

In [ ]:
freq_hz = jnp.linspace(0.01, 6.0, 150)
omega = 2.0 * jnp.pi * freq_hz

h = linear.frf(omega=omega, flowfield=None)

Extract the response of the wingtip vertical (z) displacement. The FRF returns `q`/`q_dot` over the free structural DOFs only (the root is clamped), so we need to map the tip node's global DOF index onto its position in the free-DOF ordering.

In [ ]:
n_nodes = wing.structure.n_nodes
n_dof = wing.structure.n_dof
tip_z_dof = (n_nodes - 1) * 6 + 2  # vertical translation DOF at the tip node

free_dofs = jnp.array(get_solve_dofs(n_dof=n_dof, prescribed_dofs=tuple(range(6))))
tip_idx = int(jnp.where(free_dofs == tip_z_dof)[0][0])

h_tip = h.q[:, tip_idx]

Plot the magnitude and phase of the wingtip response across the frequency sweep.

In [ ]:
fig, (ax_mag, ax_phase) = plt.subplots(2, 1, sharex=True, figsize=(7, 6))

ax_mag.semilogy(freq_hz, jnp.abs(h_tip))
ax_mag.set_ylabel("|H_tip_z| [m / (m/s)]")
ax_mag.set_title("Pazy wing tip vertical displacement response per unit gust velocity")
ax_mag.grid(True, which="both", alpha=0.3)

ax_phase.plot(freq_hz, jnp.rad2deg(jnp.angle(h_tip)))
ax_phase.set_ylabel("Phase [deg]")
ax_phase.set_xlabel("Gust frequency [Hz]")
ax_phase.grid(True, alpha=0.3)

plt.show()